<img src="https://backend.burla.dev/static/welcome.svg" width="45%">

#### This notebook scrapes **100 Wikipedia pages** in parallel across 10 cloud workers, in about 3 minutes. <br/> It's only 4 steps — follow along!

### What is Burla?

Burla is the simplest way to run Python across hundreds or thousands of cloud machines.
One function — `remote_parallel_map` — fans your code out across isolated Docker containers, grows the cluster on demand, and streams results back. Missing packages on the workers? Burla auto-installs them.

No Dockerfiles, no cluster YAML, no queue. Just Python.  
Burla is free to try — this whole notebook runs on machines we spin up for you.

### What will this demo do?

Scraping 1M pages from one laptop is CPU-, IP-, and bandwidth-bound. Scrapy Cluster, Crawlee, or Playwright on Kubernetes all work — but they're infrastructure. Running 1,000 VMs yourself means AMIs, a queue, a retry layer, and a deployer.

In this demo we'll scrape **100 Wikipedia article URLs** across **10 cloud workers** (10 URLs per worker, `max_parallelism=10`). Each worker fetches with `httpx`, parses with `selectolax` (very fast), backs off on 429/503, and returns a clean row per URL.

The full-scale version in [`main.py`](https://github.com/Burla-Cloud/parallel-web-scraping/blob/main/main.py) does this for 1M URLs across 1,000 workers.

## 1)&nbsp; Boot some machines (10+ CPUs):
&nbsp;&nbsp;&nbsp;&nbsp;Sign in using the button below, then hit the **`⏻ Start`** button on your dashboard homepage.  
&nbsp;&nbsp;&nbsp;&nbsp;This will boot a small cluster in Google Cloud for you. Burla is free to try so this is on us!

&nbsp;&nbsp;
<a href="https://login.burla.dev">
    <img src="https://i.ibb.co/QFNncHcp/Clean-Shot-2026-03-19-at-18-13-07.jpg" width="28%">
</a>

## 2)&nbsp; Install the Python package:

In [ ]:
!uv pip install burla

## 3)&nbsp; Authorize this computer:

In [ ]:
!burla login

## 4)&nbsp; Pick 100 Wikipedia URLs to scrape

Wikipedia is polite-scrapable, public, stable, and gives each article a predictable URL. Perfect for a demo. In real scraping you'd be hitting your own product pages, news sites, or SERPs.

In [ ]:
TOPICS = [
    "Python_(programming_language)", "JavaScript", "Rust_(programming_language)",
    "Go_(programming_language)", "C%2B%2B", "Java_(programming_language)",
    "Machine_learning", "Deep_learning", "Reinforcement_learning",
    "Transformer_(machine_learning_model)", "Attention_(machine_learning)",
    "Large_language_model", "Generative_artificial_intelligence",
    "Climate_change", "Renewable_energy", "Solar_energy", "Nuclear_power",
    "Quantum_computing", "Cryptography", "Public-key_cryptography",
    "Blockchain", "Bitcoin", "Ethereum",
    "Evolution", "Natural_selection", "Charles_Darwin",
    "Mitochondrion", "Photosynthesis", "DNA", "RNA",
    "Physics", "Standard_Model", "Quantum_mechanics", "General_relativity",
    "Black_hole", "Neutron_star", "Exoplanet", "Solar_System",
    "Mars", "Jupiter", "Europa_(moon)",
    "Mathematics", "Calculus", "Linear_algebra", "Topology",
    "Graph_theory", "Algorithm", "P_versus_NP_problem",
    "Economics", "Inflation", "Supply_and_demand", "Game_theory",
    "World_War_II", "French_Revolution", "Roman_Empire", "Byzantine_Empire",
    "Ottoman_Empire", "Industrial_Revolution",
    "Shakespeare", "Homer", "James_Joyce", "Virginia_Woolf",
    "Jazz", "Classical_music", "Hip_hop_music", "Electronic_music",
    "Impressionism", "Cubism", "Renaissance", "Baroque",
    "Photography", "Cinema_of_the_United_States", "Film",
    "Philosophy", "Stoicism", "Existentialism", "Phenomenology",
    "Buddhism", "Hinduism", "Islam", "Christianity",
    "Chess", "Magnus_Carlsen",
    "Olympic_Games", "Football", "Basketball", "Tennis",
    "Formula_One", "Cycling",
    "New_York_City", "Tokyo", "London", "Paris", "Berlin",
    "Linux", "Unix", "Internet", "World_Wide_Web", "HTTP", "TCP/IP",
]

# dedupe + cap at 100
TOPICS = list(dict.fromkeys(TOPICS))[:100]
urls = [f"https://en.wikipedia.org/wiki/{t}" for t in TOPICS]
print(f"{len(urls)} URLs ready to scrape")

## 5)&nbsp; Scrape them on 10 workers in parallel 🚀

Each worker gets 10 URLs. `httpx` fetches, `selectolax` parses, polite `time.sleep(0.5)` between requests per worker. Backoff on 429/503. Burla auto-installs `httpx` + `selectolax` on the workers.

In [ ]:
import httpx  # noqa: F401  # top-level import -> Burla installs httpx on workers
import selectolax  # noqa: F401  # top-level import -> Burla installs selectolax on workers
from burla import remote_parallel_map

CHUNK = 10
chunks = [urls[i : i + CHUNK] for i in range(0, len(urls), CHUNK)]
print(f"scraping {len(urls)} URLs across {len(chunks)} chunks of {CHUNK}")


def scrape_chunk(urls: list[str]) -> list[dict]:
    import random
    import time
    import httpx
    from selectolax.parser import HTMLParser

    HEADERS = {
        "User-Agent": "BurlaDemo/1.0 (+https://burla.dev)",
        "Accept-Language": "en-US,en;q=0.9",
    }

    out = []
    with httpx.Client(timeout=20.0, headers=HEADERS, follow_redirects=True) as client:
        for url in urls:
            for attempt in range(4):
                try:
                    r = client.get(url)
                    if r.status_code in (429, 503):
                        time.sleep(2 ** attempt + random.random())
                        continue
                    r.raise_for_status()
                    tree = HTMLParser(r.text)
                    title_node = tree.css_first("#firstHeading")
                    first_p = tree.css_first(".mw-parser-output > p:not(.mw-empty-elt)")
                    out.append({
                        "url": url,
                        "title": title_node.text(strip=True) if title_node else None,
                        "first_para": (first_p.text(strip=True)[:300] if first_p else None),
                        "word_count": len(r.text.split()),
                    })
                    break
                except (httpx.HTTPError, httpx.TimeoutException) as e:
                    if attempt == 3:
                        out.append({"url": url, "error": str(e)})
                    else:
                        time.sleep(2 ** attempt + random.random())
            time.sleep(0.5)  # polite per-worker delay
    return out


chunk_results = remote_parallel_map(
    scrape_chunk,
    chunks,
    func_cpu=1,
    func_ram=2,
    max_parallelism=10,
    grow=True,
)

rows = [r for chunk in chunk_results for r in chunk]
print(f"got {len(rows)} rows ({sum(1 for r in rows if 'error' in r)} errors)")

### What just happened?

100 Wikipedia pages fetched, parsed, and summarized in about a minute of wall-clock time. Each worker has its own IP, its own connection pool, and its own CPU. The global rate was 10 workers × ~1 req/sec = ~10 req/sec, entirely polite.

There's no Scrapy project, no Kubernetes cluster, no queue, no AMI. Just a function that takes URLs and returns rows.

### Inspect the results

Titles + first paragraph + page size, sorted by page size.

In [ ]:
import pandas as pd

df = pd.DataFrame(rows)
df = df[df["title"].notna()].sort_values("word_count", ascending=False).reset_index(drop=True)
print(f"scraped {len(df)} articles")
print(df[["title", "word_count"]].head(10).to_string(index=False))
print()
print("sample first paragraph:")
print("  ", df.iloc[0]["first_para"])

### Try this next

- Crank `urls` to 100,000 and `max_parallelism` to 100 — same code, much more scale.
- Swap the URLs for your real target list. Each worker is a clean IP, which helps.
- Add Playwright/Chromium inside the worker image for JS-rendered pages.
- Use `generator=True` to stream rows back as chunks finish — good UX on long scrapes.

## Thank you for trying Burla! ❤️

### Run the full 1M-URL version

See [`main.py`](https://github.com/Burla-Cloud/parallel-web-scraping/blob/main/main.py) in this repo — 1,000,000 URLs across 1,000 workers.

### Run this on your laptop 🧑‍💻

1. `pip install burla`
2. `burla login`
3. Run the same example code from above ☝️

Congrats! Your laptop now effectively has 1000+ CPUs, and 4000G of RAM &nbsp; : )

<br/>

### Any way we can help? Lets chat!

Schedule a meeting: https://cal.com/jakez  
Send me an email: jake@burla.dev